# Fixed six-head F0/I0 ensemble · preserve the0.97551 model
Train only the missing I0 ViT-B/14 heads42and123 on the exact stored3500×768 training features. Preserve the original I0seed2026 and allthree F0heads byte-for-byte. New prediction=halfmean(F0threeheads)+halfmean(I0threeheads), i.e.eachheadweight1/6. No TTA/weightsearch/pseudo-labeling/extra training images/Kaggleupload. Original0.97551CSV must reproduce exactly before any new candidate is exported. Historicalscores are user-reported references, not promised outcomes.

## 1. Environment
Choose T4 GPU. No helper module upload required. You need the saved source models, the original winning cross-backbone output folder, and the original I0 feature cache.

In [ ]:
%pip -q install scikit-learn pandas pillow kagglehub


In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
STORAGE_ROOT=Path('/content/drive/MyDrive/Car-Classification-Benchmark')
# These paths are from your reported successful runs. Edit only if you moved them.
F0_RUN_DIR=STORAGE_ROOT/'student_outputs_F0_Ensemble/20261108T091250Z_4ed04fc8'
I0_CHECKPOINT=STORAGE_ROOT/'student_outputs_I0/20261108T105209Z_7557c47a/I0_dinov2_vitb14_p3_e4_aug123_head2026.pt'
REFERENCE_RUN=STORAGE_ROOT/'student_outputs_F0_I0_Ensemble/1cafcbe05d0c'
I0_FEATURE_CACHE=None  # Optional explicit original cache path; otherwise find its exact signature below.
BATCH_SIZE=8
DOWNLOAD_FROM_KAGGLE=True
DATA_ROOT=Path('/content/car_data')
for p in [F0_RUN_DIR/'ensemble_summary.json',I0_CHECKPOINT,REFERENCE_RUN/'summary.json',REFERENCE_RUN/'F0_I0_equal_branch_ensemble_submission.csv']:
    if not p.is_file():raise FileNotFoundError(f'Missing saved source artifact: {p}. Set its real Drive path; no source models will be retrained automatically.')
if DOWNLOAD_FROM_KAGGLE:
    import kagglehub
    if not os.environ.get('KAGGLE_API_TOKEN'):
        from google.colab import userdata
        os.environ['KAGGLE_API_TOKEN']=userdata.get('KAGGLE_API_TOKEN')
    DATA_ROOT=Path(kagglehub.competition_download('idnn-swe-3052-assignment-3-flowers-classification'))


## 2. Self-contained implementation
Cache tensors/labels/normalization and dataset hashes must match the preserved I0 checkpoint exactly. No new backbone feature extraction for training; normalization is not refitted. Fresh seeds42/123 use the original AdamW/E4/35epoch/patience7recipe; seed2026 is reused, never trained. Official pretrained weights are fingerprint-checked. A pinnedGit fallback handles GitHub502/503/504 without changing source revision or disabling verification.

In [ ]:
"""Preserve the scored four-head model; add I0 heads42/123 and fixed six-head blend."""
from pathlib import Path
import argparse, gc, hashlib, json, random, subprocess, time, uuid
from urllib.error import HTTPError
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader,TensorDataset
from torchvision import datasets,transforms
from torchvision.transforms import InterpolationMode
"""Fixed half F0-three-seed, half I0-base probability blend. No fitting/upload."""
from pathlib import Path
import argparse, gc, hashlib, json, uuid
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.transforms import InterpolationMode
from PIL import Image

HUB_REF='7764ea0f912e53c92e82eb78a2a1631e92725fc8'
SEEDS=[42,123,2026]

def sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as stream:
        for chunk in iter(lambda:stream.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()

def weight_digest(model):
    h=hashlib.sha256()
    for name,value in sorted(model.state_dict().items()):
        h.update(name.encode());h.update(value.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()

def build_head(dim):
    return nn.Sequential(nn.Linear(dim,128),nn.BatchNorm1d(128),nn.ReLU(),nn.Dropout(.3),nn.Linear(128,20))

class Images(torch.utils.data.Dataset):
    def __init__(self,paths,transform):self.paths,self.transform=paths,transform
    def __len__(self):return len(self.paths)
    def __getitem__(self,index):
        with Image.open(self.paths[index]) as image:return self.transform(image.convert('RGB'))

def load_data(root):
    root=Path(root)
    train,val=[datasets.ImageFolder(root/name) for name in ('train','val')]
    assert len(train)==700 and len(val)==200 and len(train.classes)==20
    assert train.class_to_idx==val.class_to_idx
    assert all(name.startswith(f'{i:03d}_') for i,name in enumerate(train.classes))
    sample=pd.read_csv(root/'sample_submission.csv',dtype={'image_id':str})
    assert {'image_id','label'}<=set(sample.columns) and sample.image_id.is_unique and sample.image_id.notna().all()
    files=sorted(p for p in (root/'test').iterdir() if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.webp'})
    by_id={p.stem:p for p in files};ids=sample.image_id.tolist()
    assert len(by_id)==len(files) and set(by_id)==set(ids)
    return train.class_to_idx,[Path(p) for p,_ in val.samples],np.asarray(val.targets),[by_id[i] for i in ids],ids

def checked_checkpoint(path,model,dim,seed,mapping):
    p=torch.load(path,map_location='cpu',weights_only=True)
    assert p['backbone']==model and p['seed_aug']==123 and p['seed_head']==seed
    assert p['class_to_idx']==mapping
    for key in ('feature_mean','feature_std'):
        assert p[key].shape==(dim,) and torch.isfinite(p[key]).all()
    assert (p['feature_std']>0).all()
    expected={'hidden_dim':128,'use_bn':True,'dropout':.3}
    if dim==768:expected={'input_dim':768,**expected,'num_classes':20}
    assert p['head_config']==expected,'Checkpoint is not the requested E4 architecture'
    build_head(dim).load_state_dict(p['head_state_dict'],strict=True)
    return p

@torch.inference_mode()
def infer(backbone,checkpoints,paths,transform,device,batch_size=8):
    backbone.eval();assert not any(p.requires_grad for p in backbone.parameters())
    heads=[]
    for c in checkpoints:
        h=build_head(len(c['feature_mean'])).to(device)
        h.load_state_dict(c['head_state_dict'],strict=True);h.eval()
        heads.append((h,c['feature_mean'].to(device),c['feature_std'].to(device)))
    results=[[] for _ in heads]
    loader=DataLoader(Images(paths,transform),batch_size=batch_size,shuffle=False,num_workers=0)
    for index,images in enumerate(loader):
        features=backbone(images.to(device)).float()
        for j,(head,mean,std) in enumerate(heads):
            assert features.shape==(len(images),len(mean))
            p=head((features-mean)/std).softmax(1)
            assert p.shape==(len(images),20) and torch.isfinite(p).all()
            results[j].append(p.cpu())
        if index%40==0:print('PREDICT_BATCH',index+1,'/',len(loader),flush=True)
    return torch.stack([torch.cat(parts) for parts in results])

def blend_probabilities(small_heads,base):
    assert small_heads.ndim==3 and small_heads.shape[0]==3 and small_heads.shape[2]==20
    assert base.shape==small_heads.shape[1:]
    for value in (small_heads,base):
        assert torch.isfinite(value).all() and ((value>=0)&(value<=1)).all()
        assert torch.allclose(value.sum(-1),torch.ones_like(value.sum(-1)),atol=1e-5)
    small=small_heads.mean(0)
    return small,.5*small+.5*base

def metrics(target,probabilities):
    from sklearn.metrics import accuracy_score,f1_score
    pred=probabilities.argmax(1).numpy()
    return {'accuracy':float(accuracy_score(target,pred)),
        'macro_f1':float(f1_score(target,pred,labels=list(range(20)),average='macro',zero_division=0)),
        'correct':int((pred==target).sum()),'count':len(target)}

def reproduce(measured,expected,name):
    assert measured['correct']==expected['correct'] and abs(measured['accuracy']-expected['accuracy'])<1e-12, (
        f'{name} starting accuracy did not reproduce; diagnose checkpoint/source/preprocessing before exporting')

def export(path,ids,probabilities):
    pred=probabilities.argmax(1).numpy().astype(int)
    assert len(pred)==len(ids)==len(set(ids)) and ((pred>=0)&(pred<20)).all()
    pd.DataFrame({'image_id':list(ids),'label':pred}).to_csv(path,index=False,mode='x')

def run(data_root,f0_run,i0_checkpoint,storage_root,batch_size=8):
    mapping,val_paths,target,test_paths,ids=load_data(data_root)
    f0_run=Path(f0_run);i0_checkpoint=Path(i0_checkpoint)
    summary_path=f0_run/'ensemble_summary.json'
    summary=json.loads(summary_path.read_text())
    assert summary['head_seeds']==SEEDS and summary['augmentation_seed']==123
    assert summary['backbone']=='dinov2_vits14' and summary['hub_ref']==HUB_REF
    assert np.allclose(summary['weights'],[1/3]*3)
    records={r['seed']:r for r in summary['individual_validation']}
    assert set(records)==set(SEEDS)
    paths=[f0_run/Path(records[seed]['checkpoint']).name for seed in SEEDS]
    small=[checked_checkpoint(path,'dinov2_vits14',384,seed,mapping) for path,seed in zip(paths,SEEDS)]
    base=checked_checkpoint(i0_checkpoint,'dinov2_vitb14',768,2026,mapping)
    assert base['hub_ref']==HUB_REF
    source_meta=base['feature_metadata']
    source_backbone=source_meta.get('backbone',source_meta.get('model'))
    assert source_backbone=='dinov2_vitb14','I0 feature metadata identifies a different backbone'
    if 'backbone' in source_meta and 'model' in source_meta:
        assert source_meta['backbone']==source_meta['model'],'Conflicting backbone metadata'
    assert base['feature_metadata']['test_ids']==ids,'Original test ID order differs from I0 source'
    root=Path(data_root).resolve()
    for split in base['feature_metadata']['manifest']:
        # I0 uses a list of entry lists; older helpers used {'split':..., 'files':...}.
        entries=split['files'] if isinstance(split,dict) else split
        assert isinstance(entries,list),'Unrecognized I0 dataset manifest schema'
        for entry in entries:
            image=(root/entry['path']).resolve()
            assert image.is_relative_to(root) and image.is_file()
            assert sha(image)==entry['sha256'],'Dataset image differs from the I0 training/evaluation manifest'
    transform=transforms.Compose([transforms.Resize(256,interpolation=InterpolationMode.BICUBIC),
        transforms.CenterCrop(224),transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
    device='cuda' if torch.cuda.is_available() else 'cpu'
    output=Path(storage_root)/'student_outputs_F0_I0_Ensemble'/uuid.uuid4().hex[:12]
    output.mkdir(parents=True,exist_ok=False)
    print('OUTPUT',output,flush=True)
    def load(name,expected_sha,dim):
        model=torch.hub.load(f'facebookresearch/dinov2:{HUB_REF}',name,pretrained=True,trust_repo=True,skip_validation=True).to(device)
        model.requires_grad_(False).eval()
        assert model.embed_dim==dim and weight_digest(model)==expected_sha,'Pretrained backbone differs from checkpoint source'
        return model
    backbone=load('dinov2_vits14',summary['weights_sha256'],384)
    val_small=infer(backbone,small,val_paths,transform,device,batch_size)
    for index,seed in enumerate(SEEDS):reproduce(metrics(target,val_small[index]),records[seed],f'F0 seed{seed}')
    reproduce(metrics(target,val_small.mean(0)),summary['ensemble_validation'],'F0 three-seed ensemble')
    test_small=infer(backbone,small,test_paths,transform,device,batch_size)
    del backbone;gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
    backbone=load('dinov2_vitb14',base['feature_metadata']['weights_sha256'],768)
    val_base=infer(backbone,[base],val_paths,transform,device,batch_size)[0]
    reproduce(metrics(target,val_base),base['metrics'],'I0 seed2026')
    test_base=infer(backbone,[base],test_paths,transform,device,batch_size)[0]
    val_f0,val_blend=blend_probabilities(val_small,val_base)
    test_f0,test_blend=blend_probabilities(test_small,test_base)
    measured={'F0_three_seed':metrics(target,val_f0),'I0_ViTB14':metrics(target,val_base),'equal_branch_ensemble':metrics(target,val_blend)}
    export(output/'F0_three_seed_control.csv',ids,test_f0)
    export(output/'I0_ViTB14_control.csv',ids,test_base)
    export(output/'F0_I0_equal_branch_ensemble_submission.csv',ids,test_blend)
    pd.DataFrame([{'model':k,**v} for k,v in measured.items()]).to_csv(output/'validation_comparison.csv',index=False)
    np.savez_compressed(output/'validation_probabilities.npz',f0_heads=val_small.numpy(),i0=val_base.numpy(),ensemble=val_blend.numpy(),targets=target)
    provenance={'configuration':'0.5 * mean(F0 seeds42/123/2026 probabilities) + 0.5 * I0 seed2026 probabilities',
        'effective_head_weights':[1/6,1/6,1/6,1/2],'validation':measured,'submitted':False,'trained':False,'TTA':False,
        'test_labels_used':False,'test_prediction_count':len(ids),'checkpoint_sha256':{str(p):sha(p) for p in [*paths,i0_checkpoint]},
        'source_f0_summary_sha256':sha(summary_path),'hub_ref':HUB_REF,
        'selection_rule':'Fixed weights; export regardless of combined validation delta after source reproduction gates pass'}
    (output/'summary.json').write_text(json.dumps(provenance,indent=2))
    print('COMPLETE',output,json.dumps(provenance),flush=True)
    return output,provenance



SETTINGS={'epochs':35,'batch_size':64,'learning_rate':.001,'weight_decay':.001,'patience':7,'optimizer':'AdamW'}

def set_seed(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True
    torch.use_deterministic_algorithms(True)

def pinned_backbone(name,expected,dim,device):
    try:
        model=torch.hub.load(f'facebookresearch/dinov2:{HUB_REF}',name,pretrained=True,trust_repo=True,skip_validation=True)
    except HTTPError as exc:
        if exc.code not in (502,503,504):raise
        root=Path(torch.hub.get_dir())/('dinov2_git_'+HUB_REF[:12])
        if not root.exists():
            subprocess.run(['git','clone','--filter=blob:none','--no-checkout','https://github.com/facebookresearch/dinov2.git',str(root)],check=True)
        subprocess.run(['git','-C',str(root),'checkout','--detach',HUB_REF],check=True)
        assert subprocess.check_output(['git','-C',str(root),'rev-parse','HEAD'],text=True).strip()==HUB_REF
        model=torch.hub.load(str(root),name,source='local',pretrained=True)
    model.requires_grad_(False).eval()
    assert model.embed_dim==dim and weight_digest(model)==expected,'Backbone does not match original source weights'
    return model.to(device)

def feature_cache(base,cache_path,root,mapping,target,ids):
    p=torch.load(cache_path,map_location='cpu',weights_only=True)
    assert p['metadata']==base['feature_metadata']
    meta=p['metadata']
    assert meta.get('backbone',meta.get('model'))=='dinov2_vitb14'
    assert meta['seed_aug']==123 and meta['n_aug']==4 and meta['include_original'] is True
    assert meta['layout']=='view-major' and meta['feature_dim']==768
    signature=hashlib.sha256(json.dumps(p['metadata'],sort_keys=True).encode()).hexdigest()
    assert signature==base['feature_signature']
    train=datasets.ImageFolder(Path(root)/'train')
    assert train.class_to_idx==mapping and len(train)==700
    for name,count in [('X_train',3500),('X_val',200),('X_test',len(ids))]:
        x=p[name];assert x.shape==(count,768) and x.dtype==torch.float32 and torch.isfinite(x).all()
    assert torch.equal(p['y_train'],torch.tensor(train.targets).repeat(5))
    assert np.array_equal(p['y_val'].numpy(),target)
    assert p['metadata']['test_ids']==ids
    assert torch.allclose(p['X_train'].mean(0),base['feature_mean'],rtol=1e-5,atol=1e-6)
    assert torch.allclose(p['X_train'].std(0,unbiased=False).clamp_min(1e-6),base['feature_std'],rtol=1e-5,atol=1e-6)
    for split in p['metadata']['manifest']:
        entries=split['files'] if isinstance(split,dict) else split
        for entry in entries:
            path=(Path(root)/entry['path']).resolve()
            assert path.is_relative_to(Path(root).resolve()) and sha(path)==entry['sha256']
    return p

def train_missing_head(seed,base,features,output,device):
    assert seed in (42,123),'The original2026 head must never be retrained'
    assert base['training_config']==SETTINGS,'Source I0 training recipe differs from the fixed recipe'
    mean,std=base['feature_mean'],base['feature_std']
    train=(features['X_train']-mean)/std;val=(features['X_val']-mean)/std
    set_seed(seed)
    head=build_head(768).to(device)
    loader=DataLoader(TensorDataset(train,features['y_train']),batch_size=64,shuffle=True,num_workers=0,generator=torch.Generator().manual_seed(seed))
    optimizer=torch.optim.AdamW(head.parameters(),lr=.001,weight_decay=.001)
    history=[];best=-1.;best_epoch=0;weights=None;bad=0
    for epoch in range(1,36):
        head.train();loss_sum=0.
        for x,y in loader:
            optimizer.zero_grad(set_to_none=True)
            loss=torch.nn.functional.cross_entropy(head(x.to(device)),y.to(device));loss.backward();optimizer.step()
            loss_sum+=loss.item()*len(y)
        head.eval()
        with torch.inference_mode():prob=head(val.to(device)).softmax(1).cpu()
        score=metrics(features['y_val'].numpy(),prob)
        history.append({'epoch':epoch,'train_loss':loss_sum/len(train),**score})
        print('I0_HEAD',seed,'EPOCH',epoch,score,flush=True)
        if score['accuracy']>best:
            best,best_epoch,bad=score['accuracy'],epoch,0
            weights={k:v.detach().cpu().clone() for k,v in head.state_dict().items()}
        else:
            bad+=1
            if bad>=7:break
    head.load_state_dict(weights);head.eval()
    with torch.inference_mode():prob=head(val.to(device)).softmax(1).cpu()
    measured=metrics(features['y_val'].numpy(),prob)
    result={**base,'head_state_dict':weights,'seed_head':seed,'best_epoch':best_epoch,'best_accuracy':best,
            'metrics':{**measured,'seed_head':seed,'seed_aug':123,'best_epoch':best_epoch},
            'origin':'new missing I0 head; original2026 checkpoint unchanged'}
    path=Path(output)/f'I0_dinov2_vitb14_p3_e4_aug123_head{seed}.pt'
    with path.open('xb') as stream:torch.save(result,stream)
    pd.DataFrame(history).to_csv(Path(output)/f'I0_head{seed}_history.csv',index=False)
    return result,path

def average_branches(small,base):
    assert small.ndim==base.ndim==3 and small.shape==base.shape and small.shape[0]==3 and small.shape[2]==20
    for p in (small,base):
        assert torch.isfinite(p).all() and ((p>=0)&(p<=1)).all() and torch.allclose(p.sum(-1),torch.ones_like(p.sum(-1)),atol=1e-5)
    return .5*small.mean(0)+.5*base.mean(0)

def run_six(data_root,f0_run,i0_checkpoint,reference_run,cache_path,storage_root,batch_size=8):
    mapping,val_paths,target,test_paths,ids=load_data(data_root)
    f0_run=Path(f0_run);reference_run=Path(reference_run);i0_checkpoint=Path(i0_checkpoint)
    ref=json.loads((reference_run/'summary.json').read_text())
    source=json.loads((f0_run/'ensemble_summary.json').read_text())
    assert source['head_seeds']==SEEDS and source['augmentation_seed']==123 and source['hub_ref']==HUB_REF
    assert np.allclose(source['weights'],[1/3]*3)
    rows={r['seed']:r for r in source['individual_validation']};assert set(rows)==set(SEEDS)
    old_paths=[f0_run/Path(rows[s]['checkpoint']).name for s in SEEDS]+[i0_checkpoint]
    expected={Path(name).name:value for name,value in ref['checkpoint_sha256'].items()}
    assert len(expected)==4
    original_hashes={str(path):sha(path) for path in old_paths}
    for path in old_paths:assert original_hashes[str(path)]==expected[path.name],'Source checkpoint differs from the scored four-head blend'
    small=[checked_checkpoint(p,'dinov2_vits14',384,s,mapping) for p,s in zip(old_paths[:3],SEEDS)]
    base=checked_checkpoint(i0_checkpoint,'dinov2_vitb14',768,2026,mapping)
    assert base['hub_ref']==HUB_REF
    p=feature_cache(base,cache_path,data_root,mapping,target,ids)
    original_cache_sha=sha(cache_path)
    device='cuda' if torch.cuda.is_available() else 'cpu'
    output=Path(storage_root)/'student_outputs_SixHead_Ensemble'/uuid.uuid4().hex[:12];output.mkdir(parents=True,exist_ok=False)
    print('OUTPUT',output,flush=True)
    extra=[];new_paths=[]
    for seed in (42,123):
        c,path=train_missing_head(seed,base,p,output,device);extra.append(c);new_paths.append(path)
    base_heads=[*extra,base]  #42,123,2026; the source2026 object is reused verbatim.
    transform=transforms.Compose([transforms.Resize(256,interpolation=InterpolationMode.BICUBIC),transforms.CenterCrop(224),transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
    model=pinned_backbone('dinov2_vits14',source['weights_sha256'],384,device)
    vs=infer(model,small,val_paths,transform,device,batch_size)
    for i,s in enumerate(SEEDS):reproduce(metrics(target,vs[i]),rows[s],f'F0 seed{s}')
    reproduce(metrics(target,vs.mean(0)),source['ensemble_validation'],'F0 mean')
    ts=infer(model,small,test_paths,transform,device,batch_size)
    del model;gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
    model=pinned_backbone('dinov2_vitb14',base['feature_metadata']['weights_sha256'],768,device)
    vb=infer(model,base_heads,val_paths,transform,device,batch_size)
    for i,c in enumerate(base_heads):reproduce(metrics(target,vb[i]),c['metrics'],f'I0 seed{SEEDS[i]}')
    reproduce(metrics(target,.5*vs.mean(0)+.5*vb[2]),ref['validation']['equal_branch_ensemble'],'Scored four-head control')
    tb=infer(model,base_heads,test_paths,transform,device,batch_size)
    old_control=.5*ts.mean(0)+.5*tb[2]
    reference=pd.read_csv(reference_run/'F0_I0_equal_branch_ensemble_submission.csv',dtype={'image_id':str})
    assert reference.image_id.tolist()==ids
    assert np.array_equal(reference.label.to_numpy(),old_control.argmax(1).numpy()),'Reconstructed current-best predictions changed; stop before exporting a new candidate'
    val_new=average_branches(vs,vb);test_new=average_branches(ts,tb)
    export(output/'SixHead_equal_branch_ensemble_submission.csv',ids,test_new)
    export(output/'CurrentBest_four_head_control.csv',ids,old_control)
    export(output/'I0_three_seed_ensemble_control.csv',ids,tb.mean(0))
    measured={'current_best_four_head':metrics(target,.5*vs.mean(0)+.5*vb[2]),
              'I0_three_seed':metrics(target,vb.mean(0)),'six_head_equal_branch':metrics(target,val_new)}
    pd.DataFrame([{'model':name,**value} for name,value in measured.items()]).to_csv(output/'validation_comparison.csv',index=False)
    np.savez_compressed(output/'validation_probabilities.npz',small=vs.numpy(),base=vb.numpy(),six_head=val_new.numpy(),targets=target)
    for path in old_paths:assert sha(path)==original_hashes[str(path)],'Original checkpoint was modified'
    assert sha(cache_path)==original_cache_sha,'Source feature cache was modified'
    summary={'configuration':'0.5*mean(F0 seeds42/123/2026)+0.5*mean(I0 seeds42/123/2026)',
        'effective_head_weights':[1/6]*6,'validation':measured,'source_checkpoint_sha256':original_hashes,
        'new_checkpoint_sha256':{str(path):sha(path) for path in new_paths},'source_cache_sha256':original_cache_sha,
        'original_seed2026_retrained':False,'new_head_seeds':[42,123],'augmentation_seed':123,'submitted':False,
        'TTA':False,'original_scored_csv_reproduced':True,'test_labels_used':False,'prediction_count':len(ids),
        'historical_reference_Kaggle':.97551,'selection':'fixed weights; export irrespective of combined validation delta'}
    (output/'summary.json').write_text(json.dumps(summary,indent=2))
    print('COMPLETE',output,json.dumps(summary),flush=True)
    return output,summary


## 3. Find the exact I0 cache
This filename/signature must match the source seed2026 checkpoint. A cache from another preprocessing/backbone cannot be substituted. If missing, preserve/restore the original cache from the I0 run before proceeding; this notebook does not silently regenerate new training views.

In [ ]:
original_i0=torch.load(I0_CHECKPOINT,map_location='cpu',weights_only=True)
signature=original_i0['feature_signature']
filename=f'I0_dinov2_vitb14_p3_aug123_{signature}.pt'
if I0_FEATURE_CACHE is None:
    candidates=list(STORAGE_ROOT.rglob(filename))
    if len(candidates)!=1:
        print('Matching caches:',candidates)
        raise FileNotFoundError(f'Expected one exact I0 cache {filename}; found {len(candidates)}. Set I0_FEATURE_CACHE explicitly if multiple identical copies exist.')
    I0_FEATURE_CACHE=candidates[0]
else:I0_FEATURE_CACHE=Path(I0_FEATURE_CACHE)
assert I0_FEATURE_CACHE.is_file()
print('Preserving source checkpoint:',I0_CHECKPOINT)
print('Using source cache:',I0_FEATURE_CACHE)


## 4. Train only the two missing heads, verify originals, export
The original four-modelCSV and validation scores must reproduce. Equalweights were chosen before evaluation; no combined-validation improvement is required for export. Frozen backbones run sequentially to limit memory. All outputs use a newUUIDfolder.

In [ ]:
output,summary=run_six(DATA_ROOT,F0_RUN_DIR,I0_CHECKPOINT,REFERENCE_RUN,I0_FEATURE_CACHE,STORAGE_ROOT,BATCH_SIZE)
from IPython.display import display
display(pd.DataFrame([{'model':name,**value} for name,value in summary['validation'].items()]))
print('NEW CANDIDATE:',output/'SixHead_equal_branch_ensemble_submission.csv')
print('Matched0.97551control:',output/'CurrentBest_four_head_control.csv')
print('Not submitted to Kaggle.')


## 5. Use and preserve
Submit SixHead_equal_branch_ensemble_submission.csv manually as the single new candidate. Keep the scored0.97551submission and all original checkpoints. CurrentBest_four_head_control.csv is checked prediction-for-prediction against it. Preserve both new I0checkpoints and the summary in Drive. No new Kaggle score is claimed; a larger ensemble can lose. I0_three_seed_ensemble_control.csv is also exported for a separately recorded diagnostic comparison if submission limits permit.